### 1. Project Setup and Data Acquisition

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/bat-linux/.cache/kagglehub/datasets/bhavikjikadara/dog-and-cat-classification-dataset/versions/1


In [2]:
import os

cat_path = "/home/bat-linux/.cache/kagglehub/datasets/bhavikjikadara/dog-and-cat-classification-dataset/versions/1/PetImages/Cat/"
dog_path = "/home/bat-linux/.cache/kagglehub/datasets/bhavikjikadara/dog-and-cat-classification-dataset/versions/1/PetImages/Dog/"
# [Python Docs] os.listdir() returns a list of all entries
total_items_cat = len(os.listdir(cat_path))
total_items_dog = len(os.listdir(dog_path))
print(f"Total items Cat: {total_items_cat}")
print(f"Total items Dog: {total_items_dog}")


Total items Cat: 12499
Total items Dog: 12499


### 2. Data Preprocessing & Augmentation

In [3]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.5,0.5,0.5],
        std = [0.5,0.5,0.5]
    )
])

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((128,128)), 
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

In [6]:
# --- 2. Data Splitting (Run this once to set up your directory structure) ---
# Define base directories
base_data_dir = './PetImages' # The folder containing 'Cat' and 'Dog'
train_split_dir = './data_split/train' # New training data root
val_split_dir = './data_split/val'   # New validation data root

# Create the new organized directories
os.makedirs(os.path.join(train_split_dir, 'Cat'), exist_ok=True)
os.makedirs(os.path.join(train_split_dir, 'Dog'), exist_ok=True)
os.makedirs(os.path.join(val_split_dir, 'Cat'), exist_ok=True)
os.makedirs(os.path.join(val_split_dir, 'Dog'), exist_ok=True)


In [11]:
import random
import shutil
# Function to split images for a given class
def split_images(source_class_dir, train_dest_dir, val_dest_dir, split_ratio=0.8):
    images = [f for f in os.listdir(source_class_dir) if f.endswith('.jpg') or f.endswith('.png')]
    # Filter out any non-image files or corrupted files if present
    images = [img for img in images if os.path.getsize(os.path.join(source_class_dir, img)) > 0] # Exclude 0-byte files

    random.shuffle(images)
    num_train = int(len(images) * split_ratio)

    for i, img_name in enumerate(images):
        src_path = os.path.join(source_class_dir, img_name)
        if i < num_train:
            dest_path = os.path.join(train_dest_dir, img_name)
        else:
            dest_path = os.path.join(val_dest_dir, img_name)
        try:
            shutil.copy(src_path, dest_path)
        except Exception as e:
            print(f"Could not copy {src_path} to {dest_path}: {e}")

print("Starting data split...")

# Split Cat images
split_images(
    source_class_dir=os.path.join(base_data_dir, 'Cat'),
    train_dest_dir=os.path.join(train_split_dir, 'Cat'),
    val_dest_dir=os.path.join(val_split_dir, 'Cat')
)

# Split Dog images
split_images(
    source_class_dir=os.path.join(base_data_dir, 'Dog'),
    train_dest_dir=os.path.join(train_split_dir, 'Dog'),
    val_dest_dir=os.path.join(val_split_dir, 'Dog')
)

print("Data split complete!")
print(f"Training images in: {train_split_dir}")
print(f"Validation images in: {val_split_dir}")
print(f"Total training cats: {len(os.listdir(os.path.join(train_split_dir, 'Cat')))}")
print(f"Total training dogs: {len(os.listdir(os.path.join(train_split_dir, 'Dog')))}")
print(f"Total validation cats: {len(os.listdir(os.path.join(val_split_dir, 'Cat')))}")
print(f"Total validation dogs: {len(os.listdir(os.path.join(val_split_dir, 'Dog')))}")

Starting data split...
Data split complete!
Training images in: ./data_split/train
Validation images in: ./data_split/val
Total training cats: 9999
Total training dogs: 9999
Total validation cats: 2500
Total validation dogs: 2500


In [14]:
from torchvision import datasets
# --- 3. Create Datasets using ImageFolder ---
# Point ImageFolder to the newly organized directories
train_dataset = datasets.ImageFolder(root=train_split_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_split_dir, transform=val_transform)

print(f"\nTrain Dataset Size: {len(train_dataset)} images")
print(f"Validation Dataset Size: {len(val_dataset)} images")

# Print the class names and their mapping to integers (0 or 1)
print(f"Classes: {train_dataset.classes}") # e.g., ['Cat', 'Dog']
print(f"Class to index mapping: {train_dataset.class_to_idx}") # e.g., {'Cat': 0, 'Dog': 1}


Train Dataset Size: 19998 images
Validation Dataset Size: 5000 images
Classes: ['Cat', 'Dog']
Class to index mapping: {'Cat': 0, 'Dog': 1}


In [15]:
# --- 4. Create DataLoaders ---
batch_size = 64 # You can adjust this based on your GPU memory

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,          # IMPORTANT: Shuffle training data for better generalization
    num_workers=os.cpu_count() // 2 or 1 # Use some CPU cores for data loading, adjust as needed
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,         # IMPORTANT: Do NOT shuffle validation data for consistent evaluation
    num_workers=os.cpu_count() // 2 or 1
)

print(f"\nTrain DataLoader created with batch size: {batch_size}")
print(f"Validation DataLoader created with batch size: {batch_size}")

# --- You can now use train_loader and val_loader in your training loop! ---
# Example of how to get a batch:
# for images, labels in train_loader:
#     print(f"Batch images shape: {images.shape}") # e.g., torch.Size([64, 3, 128, 128])
#     print(f"Batch labels shape: {labels.shape}") # e.Size([64])
#     break # Just take one batch for demonstration


Train DataLoader created with batch size: 64
Validation DataLoader created with batch size: 64


### 3. Model Definition (CNN Architecture)

In [22]:
import torch 
import torch.nn as nn
import torch.optim as optim

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), 

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(16*16*128 , 256),
            nn.ReLU(), 
            nn.Linear(256,2)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [23]:
model = CNN()
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layer): Sequential(
    (0): Linear(in_features=32768, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=2, bias=True)
  )
)


### 4. Training Phase

In [28]:
loss = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())

In [30]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0

    for image,label in train_loader:
        optimiser.zero_grad() 

        output = model.forward(image)
        loss = criterion(output,label)
        loss.backward()
        optimiser.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} , loss = {epoch_training_loss/len(trainloader)}")

NameError: name 'trainloader' is not defined

In [ ]:
from datetime import datetime

# Get the current date and time
now = datetime.now()

# Print the full object
print(f"Current Date and Time: {now}")

In [ ]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _ , predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {(correct_labels/total_labels)*100}")